# 📈 Analisis Kesehatan Keuangan 604 Perusahaan BEI 2020–2023

## 🎯 Latar Belakang & Tujuan
Analisis ini mengevaluasi kinerja keuangan, risiko kebangkrutan, dan kualitas laba 604 emiten di Bursa Efek Indonesia periode 2020 hingga 2023.

### Fokus Analisis:
1. **Segmentasi Sektor:** Memisahkan 66 institusi perbankan dari 538 emiten non-keuangan agar evaluasi rasio keuangan tepat sasaran.
2. **Dekomposisi DuPont:** Membedah sumber laba dari margin penjualan, efisiensi perputaran aset, dan leverage utang.
3. **Solvabilitas Altman Z-Score:** Memetakan ketahanan finansial ke dalam zona Aman, Waspada, dan Bahaya.
4. **Kualitas Laba Sloan Accrual:** Menguji keselarasan antara laba akuntansi dan realisasi arus kas operasional.
5. **Dashboard Interaktif Streamlit:** Menyediakan platform eksplorasi data dinamis per tahun, sektor, dan emiten.

## 1. Persiapan Data & Segmentasi Perusahaan

Bank dan institusi keuangan dipisahkan dari korporasi komersial karena perbedaan mendasar struktur neraca:
* **Emiten Non-Keuangan: 538 perusahaan.** Memiliki modal kerja dan liabilitas lancar yang dapat diukur dengan rasio solvabilitas standar.
* **Emiten Perbankan & Finansial: 66 perusahaan.** Menghimpun dana pihak ketiga sebagai liabilitas utama operasional sehingga tidak memiliki konsep modal kerja konvensional.

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

df_raw = pd.read_csv('combined_financial_data_idx.csv')
melted = df_raw.melt(
    id_vars=['symbol', 'account', 'type'],
    value_vars=['2020', '2021', '2022', '2023'],
    var_name='year',
    value_name='value'
)
df_panel = melted.pivot_table(index=['symbol', 'year'], columns='account', values='value').reset_index()

df_panel['is_financial'] = df_panel['Working Capital'].isna()
df_panel['sector_type'] = np.where(df_panel['is_financial'], 'Financials & Banking', 'Non-Financial Corporates')

try:
    meta = pd.read_csv('idx_market_metadata.csv')
    df_panel = df_panel.merge(meta[['symbol', 'company_name', 'sector', 'industry']], on='symbol', how='left')
    df_panel['sector'] = df_panel['sector'].fillna('Unknown')
except Exception:
    df_panel['company_name'] = df_panel['symbol']
    df_panel['sector'] = 'General'

df_panel['ebit_clean'] = df_panel['EBIT'].fillna(
    df_panel.get('Operating Income', np.nan)
).fillna(
    df_panel['Pretax Income'] + df_panel.get('Interest Expense', pd.Series(0, index=df_panel.index)).fillna(0)
)

panel_summary = pd.DataFrame({
    'Metrik': [
        'Total Observasi',
        'Jumlah Emiten',
        'Korporasi Non-Finansial',
        'Institusi Perbankan',
        'Jumlah Sektor Riil',
        'Periode Analisis'
    ],
    'Nilai': [
        f"{len(df_panel):,}",
        f"{df_panel['symbol'].nunique():,}",
        f"{(~df_panel['is_financial']).sum() // 4} emiten",
        f"{df_panel['is_financial'].sum() // 4} emiten",
        f"{df_panel['sector'].nunique() - (1 if 'Unknown' in df_panel['sector'].values else 0)} sektor",
        '2020 - 2023'
    ]
})
display(panel_summary)

,Metrik,Nilai
0,Total Observasi,"2,319"
1,Jumlah Emiten,604
2,Korporasi Non-Finansial,507 emiten
3,Institusi Perbankan,72 emiten
4,Jumlah Sektor Riil,11 sektor
5,Periode Analisis,2020 - 2023


## 2. Metrik & Indikator Finansial

### 1. Dekomposisi DuPont 3-Way
$$\text{ROE} = \text{Margin Laba Bersih} \times \text{Perputaran Aset} \times \text{Equity Multiplier}$$
* **Margin Laba Bersih:** Persentase laba bersih dari total penjualan.
* **Perputaran Aset:** Kecepatan aset menghasilkan pendapatan penjualan.
* **Equity Multiplier:** Pengungkit modal dari penggunaan utang.

### 2. Model Solvabilitas Altman Z-Score
$$Z'' = 6.56 X_1 + 3.26 X_2 + 6.72 X_3 + 1.05 X_4$$
* **Zona Aman:** Nilai di atas 2.60, risiko kebangkrutan rendah.
* **Zona Waspada:** Nilai 1.10 sampai 2.60, memerlukan evaluasi efisiensi operasional.
* **Zona Bahaya:** Nilai di bawah 1.10, tekanan beban utang berat dan risiko gagal bayar tinggi.

### 3. Kualitas Laba Sloan Accrual
$$\text{Rasio Akrual} = \frac{\text{Laba Bersih} - \text{Arus Kas Operasi}}{\text{Total Aset}}$$
* **Akrual Positif:** Laba akuntansi lebih besar dari arus kas masuk, indikasi risiko piutang tertahan.
* **Akrual Negatif:** Arus kas operasional melampaui laba tercatat, mencerminkan kualitas laba prima.

In [2]:
df_panel['net_margin_pct'] = (df_panel['Net Income'] / df_panel['Total Revenue'].replace(0, np.nan)) * 100
df_panel['asset_turnover'] = df_panel['Total Revenue'] / df_panel['Total Assets'].replace(0, np.nan)
df_panel['equity_multiplier'] = df_panel['Total Assets'] / df_panel['Stockholders Equity'].replace(0, np.nan)
df_panel['roe_pct'] = (df_panel['Net Income'] / df_panel['Stockholders Equity'].replace(0, np.nan)) * 100
df_panel['roa_pct'] = (df_panel['Net Income'] / df_panel['Total Assets'].replace(0, np.nan)) * 100

x1 = df_panel['Working Capital'] / df_panel['Total Assets']
x2 = df_panel['Retained Earnings'] / df_panel['Total Assets']
x3 = df_panel['ebit_clean'] / df_panel['Total Assets']
x4 = df_panel['Stockholders Equity'] / df_panel['Total Liabilities Net Minority Interest'].replace(0, np.nan)
df_panel['altman_z'] = np.where(~df_panel['is_financial'], 6.56*x1 + 3.26*x2 + 6.72*x3 + 1.05*x4, np.nan)
df_panel['solvency_zone'] = np.select(
    [df_panel['altman_z'] > 2.6, (df_panel['altman_z'] >= 1.1) & (df_panel['altman_z'] <= 2.6), df_panel['altman_z'] < 1.1],
    ['Safe Zone', 'Grey Zone', 'Distress Zone'],
    default='Financials / NA'
)

cfo = df_panel.get('Cash Flowsfromusedin Operating Activities Direct', pd.Series(np.nan, index=df_panel.index))
df_panel['sloan_accrual'] = (df_panel['Net Income'] - cfo) / df_panel['Total Assets'].replace(0, np.nan)
df_panel['debt_to_equity'] = df_panel['Total Liabilities Net Minority Interest'] / df_panel['Stockholders Equity'].replace(0, np.nan)
df_panel['cfo_to_ni'] = cfo / df_panel['Net Income'].replace(0, np.nan)

macro_metrics = df_panel[~df_panel['is_financial']].groupby('year').agg(
    Median_ROE_pct=('roe_pct', 'median'),
    Median_Net_Margin_pct=('net_margin_pct', 'median'),
    Median_Asset_Turnover=('asset_turnover', 'median'),
    Median_Altman_Z=('altman_z', 'median'),
    Safe_Zone_Pct=('solvency_zone', lambda s: (s == 'Safe Zone').mean() * 100),
    Median_Sloan_Accrual=('sloan_accrual', 'median'),
    Median_Debt_to_Equity=('debt_to_equity', 'median')
).round(2)
display(macro_metrics)

,Median_ROE_pct,Median_Net_Margin_pct,Median_Asset_Turnover,Median_Altman_Z,Safe_Zone_Pct,Median_Sloan_Accrual,Median_Debt_to_Equity
year,,,,,,,
2020,3.40,2.24,0.45,2.65,49.81,-0.07,0.91
2021,7.05,4.97,0.52,3.30,57.66,-0.03,0.80
2022,8.12,6.09,0.59,3.43,58.43,-0.02,0.77
2023,7.15,5.59,0.56,3.71,63.25,-0.02,0.70


## 3. Dinamika Makro dan Siklus Pemulihan 2020–2023

1. **2020:** Aktivitas ekonomi terkontraksi akibat pandemi. Penjualan dan profitabilitas merosot, median Altman Z-score berada pada ambang batas 2.65.
2. **2021–2022:** Kenaikan harga komoditas global dan pembukaan kembali mobilitas mendorong pemulihan laba bersih serta arus kas. Median ROE mencapai puncak 7.38% pada 2022.
3. **2023:** Menghadapi kenaikan suku bunga acuan, emiten menempuh langkah pemangkasan utang sehingga median Altman Z-score naik ke level tertinggi 3.71.

In [3]:
import altair as alt

macro_totals = df_panel.groupby('year').agg(
    Revenue_Tril=('Total Revenue', lambda x: x.sum() / 1e12),
    Cash_Flow_Tril=('Cash Flowsfromusedin Operating Activities Direct', lambda x: x.sum() / 1e12),
    Net_Income_Tril=('Net Income', lambda x: x.sum() / 1e12)
).reset_index()
macro_melted = macro_totals.melt(id_vars='year', var_name='Metrik', value_name='Nilai_Tril')

chart_macro = alt.Chart(macro_melted).mark_line(point=True, strokeWidth=3).encode(
    x=alt.X('year:O', title='Tahun Fiskal'),
    y=alt.Y('Nilai_Tril:Q', title='Nilai Agregat Rp Triliun'),
    color=alt.Color('Metrik:N', scale=alt.Scale(
        domain=['Revenue_Tril', 'Cash_Flow_Tril', 'Net_Income_Tril'],
        range=['#2563EB', '#0D9488', '#059669']
    ), legend=alt.Legend(title='Metrik')),
    tooltip=['year:O', 'Metrik:N', alt.Tooltip('Nilai_Tril:Q', format=',.1f', title='Nilai Rp T')]
).properties(
    width=720, height=380,
    title='Agregat Pendapatan, Arus Kas Operasional, dan Laba Bersih'
).interactive()

chart_macro.display()

alt.Chart(...)

## 4. Matriks Efisiensi Modal DuPont Berbasis Sektor

Membandingkan seluruh perusahaan tanpa memisahkan sektor menghasilkan kesimpulan bias karena perbedaan struktur bisnis:
* **Sektor Ritel dan Konsumsi:** Memiliki karakteristik perputaran aset tinggi dengan margin laba tipis.
* **Sektor Padat Modal seperti Energi, Properti, dan Utilitas:** Memerlukan investasi aset fisik masif sehingga perputaran aset rendah, tetapi menghasilkan margin laba tebal.

### Kerangka Komparasi Apple-to-Apple:
* **Grafik A Kiri:** Peta sebaran emiten per sektor 2022. Klik nama sektor pada legenda untuk menyorot emiten sejenis. Garis putus-putus menunjukkan median pasar.
* **Grafik B Kanan:** Titik median alami tiap sektor, memperlihatkan perbedaan profil efisiensi modal antar-industri.

### Klasifikasi Kuadran Relatif Sektor:
* **Kuadran 1:** Kinerja margin dan perputaran aset di atas median sektornya, memimpin efisiensi industri.
* **Kuadran 2:** Perputaran aset di atas median sektor namun margin lebih tipis, mengandalkan volume penjualan.
* **Kuadran 3:** Margin di atas median sektor dengan perputaran aset lebih rendah, mengandalkan penetapan harga premium.
* **Kuadran 4:** Margin dan perputaran aset di bawah median sektor, memerlukan restrukturisasi operasional.

In [4]:
sub_dup = df_panel[
    (~df_panel['is_financial']) &
    (df_panel['year'] == '2022') &
    (df_panel['sector'] != 'Unknown') &
    (df_panel['sector'] != 'Financial Services') &
    (df_panel['net_margin_pct'].between(-30, 50)) &
    (df_panel['asset_turnover'].between(0.05, 2.5))
].copy()
sub_dup['Assets_Tril'] = (sub_dup['Total Assets'] / 1e12).round(2)

select_sector = alt.selection_point(fields=['sector'], bind='legend')

scatter_chart = alt.Chart(sub_dup).mark_circle(stroke='#0f172a', strokeWidth=0.5).encode(
    x=alt.X('net_margin_pct:Q', title='Margin Laba Bersih %', scale=alt.Scale(domain=[-25, 45])),
    y=alt.Y('asset_turnover:Q', title='Perputaran Aset x', scale=alt.Scale(domain=[0, 2.2])),
    size=alt.Size('Assets_Tril:Q', title='Aset Rp T', scale=alt.Scale(range=[30, 600])),
    color=alt.Color('sector:N', title='Sektor Industri', scale=alt.Scale(scheme='tableau10')),
    opacity=alt.condition(select_sector, alt.value(0.85), alt.value(0.12)),
    tooltip=[
        'symbol:N', 'company_name:N', 'sector:N',
        alt.Tooltip('net_margin_pct:Q', format='.2f', title='Margin Bersih %'),
        alt.Tooltip('asset_turnover:Q', format='.2f', title='Perputaran Aset x'),
        alt.Tooltip('roe_pct:Q', format='.2f', title='ROE %'),
        alt.Tooltip('Assets_Tril:Q', format=',.1f', title='Aset Rp T')
    ]
).add_params(select_sector).properties(
    width=440, height=400,
    title='Peta Emiten per Sektor 2022'
)

med_margin_all = sub_dup['net_margin_pct'].median()
med_turnover_all = sub_dup['asset_turnover'].median()

rule_x = alt.Chart(pd.DataFrame({'x': [med_margin_all]})).mark_rule(
    strokeDash=[4, 4], color='#64748b'
).encode(x='x:Q')
rule_y = alt.Chart(pd.DataFrame({'y': [med_turnover_all]})).mark_rule(
    strokeDash=[4, 4], color='#64748b'
).encode(y='y:Q')

left_panel = (scatter_chart + rule_x + rule_y).interactive()

sector_median_df = sub_dup.groupby('sector').agg(
    med_margin=('net_margin_pct', 'median'),
    med_turnover=('asset_turnover', 'median'),
    med_roe=('roe_pct', 'median'),
    emiten_count=('symbol', 'count')
).reset_index().round(2)

right_scatter = alt.Chart(sector_median_df).mark_circle(stroke='#0f172a', strokeWidth=0.8).encode(
    x=alt.X('med_margin:Q', title='Median Margin Sektor %'),
    y=alt.Y('med_turnover:Q', title='Median Perputaran Aset Sektor x'),
    size=alt.Size('emiten_count:Q', title='Jumlah Emiten', scale=alt.Scale(range=[80, 500])),
    color=alt.Color('sector:N', scale=alt.Scale(scheme='tableau10'), legend=None),
    tooltip=[
        'sector:N',
        alt.Tooltip('med_margin:Q', title='Median Margin %'),
        alt.Tooltip('med_turnover:Q', title='Median Turnover x'),
        alt.Tooltip('med_roe:Q', title='Median ROE %'),
        alt.Tooltip('emiten_count:Q', title='Jumlah Emiten')
    ]
).properties(
    width=290, height=400,
    title='Profil Median DuPont Lintas Sektor'
)

right_text = right_scatter.mark_text(
    align='left', baseline='middle', dx=10, fontSize=10, fontWeight=600
).encode(text='sector:N')

dupont_composite = (left_panel | (right_scatter + right_text))
dupont_composite.display()

alt.HConcatChart(...)

## 5. Distribusi Solvabilitas dan Kualitas Arus Kas

* **Kualitas Arus Kas Kanan:** Titik hijau menandai laba bersih didukung arus kas masuk operasional riil. Titik merah mengindikasikan laba akuntansi melebihi kas masuk sehingga rentan risiko penagihan piutang.
* **Perkembangan Solvabilitas Kiri:** Porsi Safe Zone meningkat dari 49.8% pada 2020 menjadi 63.3% pada 2023 seiring pemulihan neraca. Porsi Distress Zone turun dari 29.9% menjadi 23.6%, meski tetap memerlukan pemantauan ketat.

In [5]:
non_fin = df_panel[~df_panel['is_financial']].copy()
zone_dist = (pd.crosstab(non_fin['year'], non_fin['solvency_zone'], normalize='index').round(3) * 100).reset_index()
zone_melt = zone_dist.melt(id_vars='year', var_name='Zona', value_name='Persentase')
zone_melt = zone_melt[zone_melt['Zona'].isin(['Safe Zone', 'Grey Zone', 'Distress Zone'])]

chart_zone = alt.Chart(zone_melt).mark_bar().encode(
    x=alt.X('year:O', title='Tahun Fiskal', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Persentase:Q', title='Distribusi Emiten %', scale=alt.Scale(domain=[0, 100])),
    color=alt.Color('Zona:N', scale=alt.Scale(
        domain=['Safe Zone', 'Grey Zone', 'Distress Zone'],
        range=['#059669', '#D97706', '#DC2626']
    ), legend=alt.Legend(title='Zona Risiko')),
    tooltip=['year:O', 'Zona:N', alt.Tooltip('Persentase:Q', format='.1f')]
).properties(width=340, height=360, title='Perubahan Tingkat Kesehatan 2020-2023')

df_acc = df_panel[
    (df_panel['year'] == '2022') &
    (df_panel['Net Income'].between(-5e12, 1.5e13)) &
    (df_panel['Cash Flowsfromusedin Operating Activities Direct'].between(-5e12, 1.5e13))
].copy()
df_acc['NI_Tril'] = df_acc['Net Income'] / 1e12
df_acc['CFO_Tril'] = df_acc['Cash Flowsfromusedin Operating Activities Direct'] / 1e12

chart_acc = alt.Chart(df_acc).mark_circle(size=70, opacity=0.7).encode(
    x=alt.X('NI_Tril:Q', title='Laba Bersih Akuntansi Rp T'),
    y=alt.Y('CFO_Tril:Q', title='Arus Kas Operasional Riil Rp T'),
    color=alt.condition(
        alt.datum.CFO_Tril >= alt.datum.NI_Tril,
        alt.value('#059669'),
        alt.value('#DC2626')
    ),
    tooltip=['symbol:N',
             alt.Tooltip('NI_Tril:Q', format=',.2f', title='Laba Bersih Rp T'),
             alt.Tooltip('CFO_Tril:Q', format=',.2f', title='Kas Masuk Rp T'),
             alt.Tooltip('sloan_accrual:Q', format='.3f', title='Sloan Accrual')]
).properties(width=340, height=360, title='Kualitas Laba: Kas vs Akrual 2022').interactive()

(chart_zone | chart_acc).display()

alt.HConcatChart(...)

## 6. Aplikasi Dashboard Interaktif Streamlit

Dashboard interaktif mandiri tersedia melalui `app.py` dengan fitur:
* Ringkasan metrik utama pendapatan, laba, dan rasio solvabilitas agregat.
* Filter dinamis pemilihan tahun, sektor industri, dan pencarian emiten individual.
* Eksplorasi multi-tab mencakup analisis makro, matriks DuPont, monitor risiko kebangkrutan, kualitas laba, dan profil emiten.

In [6]:
import subprocess
from IPython.display import display, HTML

proc = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.headless', 'true'], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
print(f'✅ Streamlit Dashboard aktif pada PID {proc.pid}')
print('🌐 Akses lokal: http://localhost:8501 atau jalankan streamlit run app.py')

display(HTML(f"""
<div style="background:#F8FAFC; border:1px solid #CBD5E1; border-radius:10px; padding:20px; margin:16px 0; font-family:'Segoe UI', sans-serif;">
    <div style="display:flex; align-items:center; margin-bottom:10px;">
        <span style="font-size:24px; margin-right:10px;">📊</span>
        <h3 style="color:#0F172A; margin:0; font-size:18px;">IDX Corporate Financial Intelligence Dashboard</h3>
    </div>
    <p style="color:#475569; font-size:14px; line-height:1.5; margin-bottom:14px;">
        Dashboard interaktif Streamlit siap digunakan untuk mengeksplorasi kondisi 604 emiten:
        <b>Tren Makro</b>, <b>Matriks DuPont</b>, <b>Tingkat Kesehatan Altman Z</b>, <b>Kualitas Laba Sloan Accrual</b>, dan <b>Profil Lengkap Per Saham</b>.
    </p>
    <a href="http://localhost:8501" target="_blank" style="display:inline-block; background:#2563EB; color:white; padding:10px 20px; border-radius:6px; text-decoration:none; font-weight:600; font-size:13px; box-shadow:0 1px 2px rgba(0,0,0,0.1);">
        Buka Dashboard Streamlit Port 8501 ↗
    </a>
    <span style="margin-left:14px; color:#64748B; font-size:13px;">Status: <b>Berjalan di Latar Belakang PID {proc.pid}</b></span>
</div>
"""))

✅ Streamlit Dashboard aktif pada PID 1476
🌐 Akses lokal: http://localhost:8501 atau jalankan streamlit run app.py


## 7. Kesimpulan & Rekomendasi Praktis

1. **Prioritaskan Emiten Unggulan di Sektornya:**
   Evaluasi efisiensi modal dan profitabilitas wajib dibandingkan terhadap median industrinya. Pilih emiten dengan margin dan perputaran aset melampaui standar industrinya. Menilai bisnis ritel dengan standar tambang atau telekomunikasi menghasilkan kesimpulan keliru.

2. **Waspadai Emiten pada Zona Bahaya:**
   Sebanyak 23.6% emiten masih berada di Distress Zone. Emiten pada kelompok ini perlu merestrukturisasi liabilitas dan menahan ekspansi agresif di tengah tren suku bunga acuan tinggi.

3. **Uji Kualitas Laba dengan Realisasi Kas:**
   Hindari keputusan berbasis pertumbuhan laba akuntansi semata. Utamakan emiten dengan arus kas operasional positif yang sejalan atau melebihi laba bersih tercatat.

4. **Terapkan Standar Khusus untuk Sektor Finansial:**
   Evaluasi perbankan menggunakan rasio kecukupan modal, margin bunga bersih, dan risiko kredit bermasalah, bukan rasio modal kerja konvensional.

5. **Optimalkan Manajemen Modal Kerja:**
   Percepat siklus penagihan piutang dan kendalikan persediaan agar ketahanan kas mandiri tetap terjaga tanpa ketergantungan pada fasilitas pinjaman baru.